# 주도주 종가배팅·스윙 ALL-IN-ONE 스크리너

기존 노트북의 조건과 순서를 최대한 그대로 유지하여 한 번만 실행하고, 최종 후보를 **한 표**로 합칩니다.

### 유지한 전략 순서
1. 종가배팅 기본형  
2. 눌림목 지지  
3. 전고점 돌파  
4. 첫 거래량 장대양봉  
5. 바닥권 매물소화 대장주  
6. 60일선 기간조정 수렴  
7. 순환매 길목지키기  

### 꼭 필요한 부분만 수정
- 같은 종목 데이터를 스크리너마다 다시 받지 않고 **한 번 받아 재사용**
- FinanceDataReader의 등락률 컬럼명이 달라져도 실행되도록 호환 처리
- `KOSDAQ GLOBAL`이 누락되지 않도록 코스닥 계열 시장명 처리
- 다운로드 실패를 후보 없음으로 숨기지 않고 성공/실패 건수 표시
- 최종 결과는 한 종목당 한 행으로 합치고, 기존 전략 순서대로 정렬
- 종가배팅 레짐과 스윙 레짐을 최종 표에 함께 표시

> 신호 조건값 자체는 원본과 동일하게 유지했습니다.


In [1]:
# 최초 1회 설치
!pip install -q finance-datareader tqdm

import datetime as dt
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import FinanceDataReader as fdr
from tqdm.notebook import tqdm
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 실행 설정
# ─────────────────────────────────────────────
KST = ZoneInfo("Asia/Seoul")
TODAY = dt.datetime.now(KST).date()

MAX_WORKERS = 8       # 승선 중 불안정한 인터넷 고려: 너무 높이지 않음
RETRIES = 1
SAVE_CSV = True

# 원본 후보군 범위 유지
TOP_N_BASIC = 50
TOP_N_PULLBACK = 80
TOP_N_BREAKOUT = 80
TOP_N_FIRST_VOLUME = 100
TOP_N_MA60 = 100
TOP_N_ROTATION = 150

# 원본 조건 유지
BREAKOUT_DAYS = 60
FIRST_VOLUME_LOOKBACK = 60

print(f"✅ 설정 완료 | 한국시간 기준일: {TODAY}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.6 MB/s eta 0:00:00
✅ 설정 완료 | 한국시간 기준일: 2026-08-06


In [2]:
# ============================================================
# 1. 공통 데이터 준비
# ============================================================

def _to_number(series):
    return pd.to_numeric(series, errors="coerce")


def _main_market_mask(market_series):
    """원본 KOSPI/KOSDAQ 범위를 유지하되 KOSDAQ GLOBAL 분류 누락만 방지."""
    s = market_series.astype(str).str.upper()
    return s.eq("KOSPI") | s.str.startswith("KOSDAQ")


def _find_change_rate_column(df):
    """FinanceDataReader 버전에 따른 등락률 컬럼명 차이 대응."""
    candidates = [
        "ChgRate",
        "ChagesRatio",   # 일부 FDR KRX 캐시에서 사용하는 명칭
        "ChangesRatio",
        "ChangeRate",
    ]
    for col in candidates:
        if col in df.columns:
            return col

    # 최후 fallback: 현재가와 전일대비 금액이 있으면 등락률 역산
    close_col = next((c for c in ["Close"] if c in df.columns), None)
    change_col = next((c for c in ["Change", "Changes"] if c in df.columns), None)
    if close_col and change_col:
        prev_close = _to_number(df[close_col]) - _to_number(df[change_col])
        df["_CalculatedChangeRate"] = np.where(
            prev_close != 0,
            _to_number(df[change_col]) / prev_close * 100,
            np.nan,
        )
        return "_CalculatedChangeRate"

    return None


def load_listing():
    krx = fdr.StockListing("KRX").copy()

    required = {"Code", "Name", "Market", "Amount"}
    missing = required - set(krx.columns)
    if missing:
        raise RuntimeError(f"KRX 종목목록 필수 컬럼 누락: {sorted(missing)}")

    krx["Code"] = krx["Code"].astype(str).str.zfill(6)
    krx["Amount"] = _to_number(krx["Amount"])

    change_col = _find_change_rate_column(krx)
    if change_col is not None:
        krx["_ChangeRate"] = _to_number(krx[change_col])
    else:
        krx["_ChangeRate"] = np.nan

    krx = krx.dropna(subset=["Code", "Name", "Amount"]).copy()
    main = krx[_main_market_mask(krx["Market"])].copy()

    return krx, main, change_col


krx_all, krx_main, CHANGE_RATE_SOURCE = load_listing()

top50 = krx_main.nlargest(TOP_N_BASIC, "Amount").copy()
top80 = krx_main.nlargest(max(TOP_N_PULLBACK, TOP_N_BREAKOUT), "Amount").copy()
top100 = krx_main.nlargest(max(TOP_N_FIRST_VOLUME, TOP_N_MA60), "Amount").copy()
top150_all = krx_all.nlargest(TOP_N_ROTATION, "Amount").copy()

# ⑤ 바닥권 매물소화의 원본 1차 필터
bottom_leader_universe = krx_main[
    (krx_main["Amount"] >= 10_000_000_000)
    & (krx_main["_ChangeRate"] >= 3.0)
].copy()

# 모든 전략 후보 종목을 합친 뒤, 가격 데이터는 종목당 한 번만 다운로드
universe_frames = [
    top50,
    top80,
    top100,
    top150_all,
    bottom_leader_universe,
]
universe = (
    pd.concat(universe_frames, ignore_index=True)
    .drop_duplicates(subset=["Code"], keep="first")
    .copy()
)

# 가장 긴 원본 분석기간(250일)을 한 번에 사용
STOCK_START = (TODAY - dt.timedelta(days=250)).strftime("%Y-%m-%d")


def download_stock(code):
    last_error = None
    for attempt in range(RETRIES + 1):
        try:
            df = fdr.DataReader(code, start=STOCK_START)
            if df is None or df.empty:
                raise ValueError("빈 가격 데이터")

            needed = {"Open", "High", "Low", "Close", "Volume"}
            missing = needed - set(df.columns)
            if missing:
                raise ValueError(f"가격 필수 컬럼 누락: {sorted(missing)}")

            df = df.sort_index()
            df = df[~df.index.duplicated(keep="last")].copy()

            # 모든 전략이 공통 사용
            df["MA5"] = df["Close"].rolling(5).mean()
            df["MA20"] = df["Close"].rolling(20).mean()
            df["MA60"] = df["Close"].rolling(60).mean()
            df["MA120"] = df["Close"].rolling(120).mean()

            # 원본별 계산 방식을 모두 보존
            df["Vol_MA20_Prev"] = df["Volume"].shift(1).rolling(20).mean()
            df["Vol_MA20_IncludingToday"] = df["Volume"].rolling(20).mean()
            return df, None

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            if attempt < RETRIES:
                time.sleep(1 + attempt)

    return None, last_error


price_cache = {}
download_errors = {}

print(f"📥 공통 가격 데이터 다운로드: {len(universe)}종목")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_stock, code): code
        for code in universe["Code"].tolist()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="가격 데이터 수집"):
        code = futures[future]
        try:
            df, err = future.result()
            if df is not None:
                price_cache[code] = df
            else:
                download_errors[code] = err or "알 수 없는 오류"
        except Exception as e:
            download_errors[code] = f"{type(e).__name__}: {e}"


# ============================================================
# 2. 원본 전략 조건 — 조건값과 순서 유지
# ============================================================

STRATEGY_ORDER = [
    "① 종가배팅 기본형",
    "② 눌림목 지지",
    "③ 전고점 돌파",
    "④ 첫 거래량 장대양봉",
    "⑤ 바닥권 매물소화",
    "⑥ 60일선 기간조정",
    "⑦ 순환매 길목",
]

signals = {name: {} for name in STRATEGY_ORDER}


def _valid_numbers(*values):
    return all(pd.notna(v) and np.isfinite(v) for v in values)


# ① 종가배팅 기본형 — 원본 top 50
for _, row in top50.iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < 60:
        continue

    latest = df.iloc[-1]
    close = latest["Close"]
    volume = latest["Volume"]
    ma5 = latest["MA5"]
    ma20 = latest["MA20"]
    ma60 = latest["MA60"]
    vol_ma20 = latest["Vol_MA20_Prev"]

    if not _valid_numbers(close, volume, ma5, ma20, ma60, vol_ma20) or vol_ma20 <= 0:
        continue

    if not (close > ma20 and close > ma60 and ma5 > ma20):
        continue
    if volume < vol_ma20 * 3.0:
        continue

    signals[STRATEGY_ORDER[0]][code] = (
        f"✅ 거래량 {volume/vol_ma20*100:.0f}% · 20일선 {int(ma20):,}"
    )


# ② 눌림목 지지 — 원본 top 80
for _, row in top80.head(TOP_N_PULLBACK).iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < 65:
        continue

    latest = df.iloc[-1]
    close, high, low = latest["Close"], latest["High"], latest["Low"]
    volume = latest["Volume"]
    ma5, ma20, ma60 = latest["MA5"], latest["MA20"], latest["MA60"]
    vol_ma20 = latest["Vol_MA20_Prev"]
    ma60_5ago = df["MA60"].iloc[-6]
    recent_high = df["High"].iloc[-15:].max()  # 원본과 동일: 오늘 포함

    if not _valid_numbers(
        close, high, low, volume, ma5, ma20, ma60,
        vol_ma20, ma60_5ago, recent_high
    ) or vol_ma20 <= 0 or recent_high <= 0:
        continue

    if not (ma20 > ma60 and ma60 > ma60_5ago):
        continue
    if close < ma20:
        continue
    if not (close <= ma5 * 1.03):  # 원본 조건 그대로
        continue

    pullback = (recent_high - close) / recent_high
    if not (0.03 <= pullback <= 0.15):
        continue
    if volume > vol_ma20 * 1.2:
        continue

    candle_range = high - low
    if candle_range <= 0:
        continue
    close_pos = (close - low) / candle_range
    if close_pos < 0.4:
        continue

    signals[STRATEGY_ORDER[1]][code] = (
        f"✅ 조정 {pullback*100:.1f}% · 거래량 {volume/vol_ma20*100:.0f}%"
    )


# ③ 전고점 돌파 — 원본 top 80
for _, row in top80.head(TOP_N_BREAKOUT).iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < BREAKOUT_DAYS + 5:
        continue

    latest = df.iloc[-1]
    close, high, low, op = (
        latest["Close"], latest["High"], latest["Low"], latest["Open"]
    )
    volume = latest["Volume"]
    ma20, ma60 = latest["MA20"], latest["MA60"]
    vol_ma20 = latest["Vol_MA20_Prev"]
    prev_high_close = df["Close"].iloc[-(BREAKOUT_DAYS + 1):-1].max()

    if not _valid_numbers(
        close, high, low, op, volume, ma20, ma60,
        vol_ma20, prev_high_close
    ) or vol_ma20 <= 0 or prev_high_close <= 0:
        continue

    if not (close > prev_high_close):
        continue
    if not (close > ma20 and ma20 > ma60):
        continue
    if not (close > op):
        continue

    candle_range = high - low
    if candle_range <= 0:
        continue
    close_pos = (close - low) / candle_range
    if close_pos < 0.7:
        continue
    if volume < vol_ma20 * 1.5:
        continue

    breakout_pct = (close - prev_high_close) / prev_high_close
    signals[STRATEGY_ORDER[2]][code] = (
        f"✅ 돌파 +{breakout_pct*100:.1f}% · 종가위치 {close_pos*100:.0f}%"
        f" · 거래량 {volume/vol_ma20*100:.0f}%"
    )


# ④ 첫 거래량 장대양봉 — 원본 top 100
for _, row in top100.head(TOP_N_FIRST_VOLUME).iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < FIRST_VOLUME_LOOKBACK + 25:
        continue

    latest = df.iloc[-1]
    close, high, low, op = (
        latest["Close"], latest["High"], latest["Low"], latest["Open"]
    )
    prev_close = df["Close"].iloc[-2]
    volume = latest["Volume"]
    ma60 = latest["MA60"]
    vol_ma20 = latest["Vol_MA20_Prev"]

    if not _valid_numbers(
        close, high, low, op, prev_close, volume, ma60, vol_ma20
    ) or vol_ma20 <= 0 or prev_close <= 0 or op <= 0:
        continue

    if volume < vol_ma20 * 3.0:
        continue

    past = df.iloc[-(FIRST_VOLUME_LOOKBACK + 1):-1]
    past_explosion = (
        past["Volume"] > past["Vol_MA20_Prev"] * 3.0
    ).sum()
    if past_explosion > 0:
        continue

    day_return = (close - prev_close) / prev_close
    if day_return < 0.06:
        continue

    body = (close - op) / op
    if body < 0.04:
        continue

    candle_range = high - low
    if candle_range <= 0:
        continue
    close_pos = (close - low) / candle_range
    if close_pos < 0.75:
        continue

    if close < ma60:
        continue

    signals[STRATEGY_ORDER[3]][code] = (
        f"✅ 상승 +{day_return*100:.1f}% · 몸통 +{body*100:.1f}%"
        f" · 거래량 {volume/vol_ma20*100:.0f}%"
    )


# ⑤ 바닥권 매물소화 — 원본 거래대금 100억+ / 등락률 3%+
for _, row in bottom_leader_universe.iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < 60:
        continue

    latest = df.iloc[-1]
    close = latest["Close"]
    volume = latest["Volume"]
    ma5, ma20, ma60 = latest["MA5"], latest["MA20"], latest["MA60"]
    avg_vol_5d = df["Volume"].iloc[-6:-1].mean()
    min_price_120d = df["Close"].tail(120).min()
    chg_rate = row["_ChangeRate"]

    if not _valid_numbers(
        close, volume, ma5, ma20, ma60,
        avg_vol_5d, min_price_120d, chg_rate
    ) or avg_vol_5d <= 0 or min_price_120d <= 0:
        continue

    price_from_bottom = (close / min_price_120d - 1) * 100
    cond_vol = volume > avg_vol_5d * 5.0
    cond_ma = close > ma5 and close > ma20 and close > ma60
    cond_bottom = price_from_bottom <= 60.0

    if cond_vol and cond_ma and cond_bottom:
        signals[STRATEGY_ORDER[4]][code] = (
            f"✅ 등락 {chg_rate:+.2f}% · 바닥대비 {price_from_bottom:.1f}%"
            f" · 5일평균대비 거래량 {volume/avg_vol_5d*100:.0f}%"
        )


# ⑥ 60일선 기간조정 — 원본 top 100, 손바뀜은 표시만 하고 필수조건 아님
for _, row in top100.head(TOP_N_MA60).iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < 70:
        continue

    latest = df.iloc[-1]
    close = latest["Close"]
    ma60 = latest["MA60"]

    if not _valid_numbers(close, ma60) or ma60 <= 0:
        continue
    if not (ma60 * 0.96 <= close <= ma60 * 1.04):
        continue

    recent_5 = df.iloc[-5:]
    mean_close = recent_5["Close"].mean()
    price_std = recent_5["Close"].std()
    if not _valid_numbers(mean_close, price_std) or mean_close <= 0:
        continue
    if price_std / mean_close > 0.03:
        continue

    recent_20 = df.iloc[-20:]
    has_handshake = (
        recent_20["Volume"]
        > recent_20["Vol_MA20_Prev"] * 4.0
    ).any()

    distance = (close / ma60 - 1) * 100
    handshake_text = "손바뀜 포착🔥" if has_handshake else "손바뀜 미포착"
    signals[STRATEGY_ORDER[5]][code] = (
        f"✅ 60일선거리 {distance:+.1f}% · {handshake_text}"
    )


# ⑦ 순환매 길목 — 원본 KRX 전체 거래대금 top 150
for _, row in top150_all.iterrows():
    code = row["Code"]
    df = price_cache.get(code)
    if df is None or len(df) < 130:
        continue

    latest = df.iloc[-1]
    close = latest["Close"]
    volume = latest["Volume"]
    ma60, ma120 = latest["MA60"], latest["MA120"]
    vol_ma20 = latest["Vol_MA20_IncludingToday"]  # 원본과 동일: 오늘 포함

    if not _valid_numbers(close, volume, ma60, ma120, vol_ma20) or vol_ma20 <= 0:
        continue

    is_near_ma60 = ma60 * 0.97 <= close <= ma60 * 1.03
    is_near_ma120 = ma120 * 0.97 <= close <= ma120 * 1.03
    if not (is_near_ma60 or is_near_ma120):
        continue
    if volume > vol_ma20 * 0.6:
        continue

    recent_5 = df.iloc[-5:]
    mean_close = recent_5["Close"].mean()
    price_std = recent_5["Close"].std()
    if not _valid_numbers(mean_close, price_std) or mean_close <= 0:
        continue
    if price_std / mean_close > 0.025:
        continue

    support_line = "120일선" if is_near_ma120 else "60일선"
    signals[STRATEGY_ORDER[6]][code] = (
        f"✅ {support_line} · 거래량 {volume/vol_ma20*100:.0f}%"
    )


# ============================================================
# 3. 시장 레짐 — 원본 점수체계 유지
# ============================================================

INDEX_START_CLOSE = (TODAY - dt.timedelta(days=200)).strftime("%Y-%m-%d")
INDEX_START_SWING = (TODAY - dt.timedelta(days=300)).strftime("%Y-%m-%d")


def load_index(symbol, start):
    df = fdr.DataReader(symbol, start=start).sort_index().copy()
    if df is None or df.empty:
        raise ValueError(f"{symbol} 지수 데이터 없음")
    return df


def closing_regime():
    try:
        kospi = load_index("KS11", INDEX_START_CLOSE)
        kosdaq = load_index("KQ11", INDEX_START_CLOSE)

        for df in [kospi, kosdaq]:
            df["MA5"] = df["Close"].rolling(5).mean()
            df["MA20"] = df["Close"].rolling(20).mean()
            df["MA60"] = df["Close"].rolling(60).mean()

        score = 0
        max_score = 0
        reasons = []

        def check(condition, weight, pass_text, fail_text):
            nonlocal score, max_score
            max_score += weight
            if condition:
                score += weight
                reasons.append(f"✅ {pass_text} (+{weight})")
            else:
                reasons.append(f"❌ {fail_text} (0)")

        def trend_status(df, name, weight_trend, weight_slope):
            latest = df.iloc[-1]
            close, ma20, ma60 = latest["Close"], latest["MA20"], latest["MA60"]
            ma60_5ago = df["MA60"].iloc[-6]
            check(
                close > ma20 and close > ma60,
                weight_trend,
                f"{name} 추세 양호",
                f"{name} 추세 약함",
            )
            check(
                ma60 > ma60_5ago,
                weight_slope,
                f"{name} 60일선 우상향",
                f"{name} 60일선 횡보/하락",
            )

        trend_status(kospi, "코스피", 25, 10)
        trend_status(kosdaq, "코스닥", 30, 15)

        recent = kosdaq["Volume"].iloc[-5:].mean()
        base = kosdaq["Volume"].iloc[-25:-5].mean()
        check(
            recent > base,
            10,
            f"코스닥 거래량 증가 ({recent/base*100:.0f}%)",
            f"코스닥 거래량 위축 ({recent/base*100:.0f}%)",
        )

        vkospi = None
        for symbol in ["VKOSPI", "KSVKOSPI"]:
            try:
                test = load_index(symbol, INDEX_START_CLOSE)
                if len(test) > 20:
                    vkospi = test
                    break
            except Exception:
                pass

        if vkospi is not None:
            vk_now = vkospi["Close"].iloc[-1]
            vk_ma20 = vkospi["Close"].iloc[-20:].mean()
            check(
                vk_now < vk_ma20 * 1.3 and vk_now < 30,
                10,
                f"변동성 안정 (VKOSPI {vk_now:.1f})",
                f"변동성 과열 (VKOSPI {vk_now:.1f})",
            )
        else:
            reasons.append("⚠️ VKOSPI 미확보 — 원본과 동일하게 항목 제외")

        pct = score / max_score * 100 if max_score else np.nan
        if pct >= 70:
            verdict = "🟢 추천"
        elif pct >= 45:
            verdict = "🟡 주의"
        else:
            verdict = "🔴 비추천"

        return {
            "pct": pct,
            "label": f"{verdict} {pct:.0f}%",
            "reasons": reasons,
            "date": str(max(kospi.index[-1], kosdaq.index[-1]).date()),
        }
    except Exception as e:
        return {
            "pct": np.nan,
            "label": "⚠️ 확인실패",
            "reasons": [f"{type(e).__name__}: {e}"],
            "date": "-",
        }


def swing_regime():
    try:
        kospi = load_index("KS11", INDEX_START_SWING)
        kosdaq = load_index("KQ11", INDEX_START_SWING)

        for df in [kospi, kosdaq]:
            df["MA20"] = df["Close"].rolling(20).mean()
            df["MA60"] = df["Close"].rolling(60).mean()
            df["MA120"] = df["Close"].rolling(120).mean()
            df["Ret"] = df["Close"].pct_change()

        score = 0
        max_score = 0
        reasons = []
        regime_warnings = []

        def check(condition, weight, pass_text, fail_text, warn_if_fail=None):
            nonlocal score, max_score
            max_score += weight
            if condition:
                score += weight
                reasons.append(f"✅ {pass_text} (+{weight})")
            else:
                reasons.append(f"❌ {fail_text} (0)")
                if warn_if_fail:
                    regime_warnings.append(warn_if_fail)

        def swing_trend(df, name, w_mid, w_long, w_slope):
            latest = df.iloc[-1]
            close, ma60, ma120 = latest["Close"], latest["MA60"], latest["MA120"]
            ma60_10ago = df["MA60"].iloc[-11]

            check(
                close > ma60, w_mid,
                f"{name} 60일선 위", f"{name} 60일선 이탈",
                f"{name} 60일선 이탈",
            )
            check(
                close > ma120, w_long,
                f"{name} 120일선 위", f"{name} 120일선 이탈",
            )
            check(
                ma60 > ma60_10ago, w_slope,
                f"{name} 60일선 우상향", f"{name} 60일선 꺾임",
                f"{name} 60일선 기울기 하락",
            )

        swing_trend(kospi, "코스피", 15, 10, 10)
        swing_trend(kosdaq, "코스닥", 15, 10, 10)

        for df, name in [(kospi, "코스피"), (kosdaq, "코스닥")]:
            latest = df.iloc[-1]
            check(
                latest["MA60"] > latest["MA120"],
                5,
                f"{name} 중기 정배열",
                f"{name} 역배열/혼조",
            )

        recent_vol = kospi["Ret"].iloc[-20:].std() * (252 ** 0.5) * 100
        base_vol = kospi["Ret"].iloc[-60:-20].std() * (252 ** 0.5) * 100
        check(
            recent_vol < base_vol * 1.3 and recent_vol < 30,
            10,
            f"변동성 안정 ({recent_vol:.1f}%)",
            f"변동성 상승 ({recent_vol:.1f}%)",
            "변동성 급등",
        )

        pct = score / max_score * 100 if max_score else np.nan
        if pct >= 70:
            verdict = "🟢 추천"
        elif pct >= 45:
            verdict = "🟡 선별"
        else:
            verdict = "🔴 관망"

        return {
            "pct": pct,
            "label": f"{verdict} {pct:.0f}%",
            "reasons": reasons,
            "warnings": regime_warnings,
            "date": str(max(kospi.index[-1], kosdaq.index[-1]).date()),
        }
    except Exception as e:
        return {
            "pct": np.nan,
            "label": "⚠️ 확인실패",
            "reasons": [f"{type(e).__name__}: {e}"],
            "warnings": [],
            "date": "-",
        }


closing_info = closing_regime()
swing_info = swing_regime()


# ============================================================
# 4. 한 종목 한 행으로 최종 통합
# ============================================================

listing_meta = (
    krx_all.sort_values("Amount", ascending=False)
    .drop_duplicates("Code")
    .set_index("Code")
)

matched_codes = set()
for strategy_name in STRATEGY_ORDER:
    matched_codes.update(signals[strategy_name].keys())

rows = []
for code in matched_codes:
    meta = listing_meta.loc[code] if code in listing_meta.index else None
    df = price_cache.get(code)

    matched = [
        strategy_name
        for strategy_name in STRATEGY_ORDER
        if code in signals[strategy_name]
    ]
    first_strategy = matched[0]
    priority_no = STRATEGY_ORDER.index(first_strategy) + 1

    if df is not None and not df.empty:
        close = df["Close"].iloc[-1]
        data_date = str(pd.Timestamp(df.index[-1]).date())
    else:
        close = np.nan
        data_date = "-"

    amount = float(meta["Amount"]) if meta is not None and pd.notna(meta["Amount"]) else np.nan
    name = str(meta["Name"]) if meta is not None else code
    market = str(meta["Market"]) if meta is not None else "-"

    row_out = {
        "_우선순위": priority_no,
        "_거래대금정렬": amount,
        "첫 포착": first_strategy,
        "종목코드": code,
        "종목명": name,
        "시장": market,
        "현재가": f"{int(close):,}" if pd.notna(close) else "-",
        "거래대금(억)": round(amount / 100_000_000) if pd.notna(amount) else np.nan,
        "통과수": len(matched),
    }

    for strategy_name in STRATEGY_ORDER:
        row_out[strategy_name] = signals[strategy_name].get(code, "—")

    row_out["종가레짐"] = closing_info["label"]
    row_out["스윙레짐"] = swing_info["label"]
    row_out["가격기준일"] = data_date
    rows.append(row_out)

sort_cols = ["_우선순위", "_거래대금정렬"]
if rows:
    final_all_in_one = (
        pd.DataFrame(rows)
        .sort_values(sort_cols, ascending=[True, False])
        .drop(columns=sort_cols)
        .reset_index(drop=True)
    )
else:
    final_all_in_one = pd.DataFrame(
        columns=[
            "첫 포착", "종목코드", "종목명", "시장", "현재가",
            "거래대금(억)", "통과수", *STRATEGY_ORDER,
            "종가레짐", "스윙레짐", "가격기준일",
        ]
    )

# 실행 상태 요약
success_count = len(price_cache)
total_count = len(universe)
fail_count = len(download_errors)
fail_rate = fail_count / total_count * 100 if total_count else 0

print("\n" + "=" * 78)
print("📌 ALL-IN-ONE 실행 요약")
print(f"• 종목목록: 전체 {len(krx_all):,}개 / 주시장 {len(krx_main):,}개")
print(f"• 등락률 원본 컬럼: {CHANGE_RATE_SOURCE or '미확보'}")
print(f"• 가격 다운로드: 성공 {success_count:,} / 대상 {total_count:,} / 실패 {fail_count:,} ({fail_rate:.1f}%)")
print(f"• 종가배팅 레짐: {closing_info['label']} | 지수 기준일 {closing_info['date']}")
print(f"• 스윙 레짐: {swing_info['label']} | 지수 기준일 {swing_info['date']}")
print(f"• 최종 후보: {len(final_all_in_one):,}개")
if fail_rate >= 5:
    print("⚠️ 다운로드 실패율이 5% 이상입니다. 결과를 사용하기 전에 오류 목록을 확인하세요.")
print("=" * 78)

print("\n🚦 종가배팅 레짐 근거")
for text in closing_info["reasons"]:
    print(" ", text)

print("\n🌊 스윙 레짐 근거")
for text in swing_info["reasons"]:
    print(" ", text)
if swing_info.get("warnings"):
    print(" ⚠️", " / ".join(swing_info["warnings"]))

# 파일 저장
date_tag = TODAY.strftime("%Y%m%d")
result_filename = f"주도주_ALL_IN_ONE_결과_{date_tag}.csv"
error_filename = f"주도주_ALL_IN_ONE_오류_{date_tag}.csv"

if SAVE_CSV:
    final_all_in_one.to_csv(result_filename, index=False, encoding="utf-8-sig")
    if download_errors:
        error_rows = []
        meta_names = listing_meta["Name"].to_dict() if "Name" in listing_meta.columns else {}
        for code, error in download_errors.items():
            error_rows.append({
                "종목코드": code,
                "종목명": meta_names.get(code, "-"),
                "오류": error,
            })
        pd.DataFrame(error_rows).to_csv(error_filename, index=False, encoding="utf-8-sig")

# 최종 결과는 한 표만 표시
print("\n📊 최종 ALL-IN-ONE 결과")
if final_all_in_one.empty:
    print("오늘 7개 전략 중 하나 이상을 통과한 종목이 없습니다.")
    display(final_all_in_one)
else:
    styled = (
        final_all_in_one.style
        .hide(axis="index")
        .set_properties(**{
            "text-align": "center",
            "white-space": "nowrap",
            "font-size": "12px",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("position", "sticky"),
                    ("top", "0"),
                    ("background-color", "#f2f2f2"),
                    ("z-index", "2"),
                    ("text-align", "center"),
                    ("white-space", "nowrap"),
                ],
            },
            {
                "selector": "td",
                "props": [("padding", "6px 8px")],
            },
        ])
    )
    html = styled.to_html()
    display(HTML(
        '<div style="overflow-x:auto; max-height:720px; border:1px solid #ddd;">'
        + html
        + '</div>'
    ))

print(f"\n💾 결과 CSV: {result_filename}")
if download_errors:
    print(f"🧾 오류 CSV: {error_filename}")


📥 공통 가격 데이터 다운로드: 187종목


가격 데이터 수집:   0%|          | 0/187 [00:00<?, ?it/s]


📌 ALL-IN-ONE 실행 요약
• 종목목록: 전체 2,872개 / 주시장 2,763개
• 등락률 원본 컬럼: ChagesRatio
• 가격 다운로드: 성공 187 / 대상 187 / 실패 0 (0.0%)
• 종가배팅 레짐: 🔴 비추천 0% | 지수 기준일 2026-08-06
• 스윙 레짐: 🔴 관망 6% | 지수 기준일 2026-08-06
• 최종 후보: 9개

🚦 종가배팅 레짐 근거
  ❌ 코스피 추세 약함 (0)
  ❌ 코스피 60일선 횡보/하락 (0)
  ❌ 코스닥 추세 약함 (0)
  ❌ 코스닥 60일선 횡보/하락 (0)
  ❌ 코스닥 거래량 위축 (96%) (0)
  ⚠️ VKOSPI 미확보 — 원본과 동일하게 항목 제외

🌊 스윙 레짐 근거
  ❌ 코스피 60일선 이탈 (0)
  ❌ 코스피 120일선 이탈 (0)
  ❌ 코스피 60일선 꺾임 (0)
  ❌ 코스닥 60일선 이탈 (0)
  ❌ 코스닥 120일선 이탈 (0)
  ❌ 코스닥 60일선 꺾임 (0)
  ✅ 코스피 중기 정배열 (+5)
  ❌ 코스닥 역배열/혼조 (0)
  ❌ 변동성 상승 (102.2%) (0)
 ⚠️ 코스피 60일선 이탈 / 코스피 60일선 기울기 하락 / 코스닥 60일선 이탈 / 코스닥 60일선 기울기 하락 / 변동성 급등

📊 최종 ALL-IN-ONE 결과


첫 포착,종목코드,종목명,시장,현재가,거래대금(억),통과수,① 종가배팅 기본형,② 눌림목 지지,③ 전고점 돌파,④ 첫 거래량 장대양봉,⑤ 바닥권 매물소화,⑥ 60일선 기간조정,⑦ 순환매 길목,종가레짐,스윙레짐,가격기준일
① 종가배팅 기본형,010130,고려아연,KOSPI,"1,234,000",1060,1,"✅ 거래량 503% · 20일선 1,018,850",—,—,—,—,—,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
① 종가배팅 기본형,051900,LG생활건강,KOSPI,"324,000",903,2,"✅ 거래량 319% · 20일선 261,750",—,✅ 돌파 +10.8% · 종가위치 88% · 거래량 319%,—,—,—,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
② 눌림목 지지,207940,삼성바이오로직스,KOSPI,"1,514,000",730,1,—,✅ 조정 3.9% · 거래량 71%,—,—,—,—,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
② 눌림목 지지,000810,삼성화재,KOSPI,"648,000",713,2,—,✅ 조정 7.6% · 거래량 63%,—,—,—,✅ 60일선거리 +3.5% · 손바뀜 미포착,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
⑤ 바닥권 매물소화,271980,제일약품,KOSPI,"10,770",114,1,—,—,—,—,✅ 등락 +3.66% · 바닥대비 16.7% · 5일평균대비 거래량 4863%,—,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
⑥ 60일선 기간조정,035720,카카오,KOSPI,"38,300",895,1,—,—,—,—,—,✅ 60일선거리 +0.5% · 손바뀜 미포착,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
⑥ 60일선 기간조정,003490,대한항공,KOSPI,"27,500",699,1,—,—,—,—,—,✅ 60일선거리 +3.3% · 손바뀜 미포착,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
⑥ 60일선 기간조정,033780,KT&G,KOSPI,"182,700",527,1,—,—,—,—,—,✅ 60일선거리 +2.0% · 손바뀜 미포착,—,🔴 비추천 0%,🔴 관망 6%,2026-08-06
⑥ 60일선 기간조정,096770,SK이노베이션,KOSPI,"108,600",525,2,—,—,—,—,—,✅ 60일선거리 -1.9% · 손바뀜 미포착,✅ 60일선 · 거래량 52%,🔴 비추천 0%,🔴 관망 6%,2026-08-06



💾 결과 CSV: 주도주_ALL_IN_ONE_결과_20260806.csv
